In [0]:
BRONZE = "olist_project.bronze"
SILVER = "olist_project.silver"
GOLD = "olist_project.gold"

In [0]:
# orders — most frequently joined table, Z-ORDER on commonly filtered columns
spark.sql(f"OPTIMIZE {SILVER}.orders ZORDER BY (order_status, order_purchase_timestamp)")
# order_items — frequently joined with orders and products
spark.sql(f"OPTIMIZE {SILVER}.order_items ZORDER BY (order_id, product_id)")
# order_payments — frequently joined with orders
spark.sql(f"OPTIMIZE {SILVER}.order_payments ZORDER BY (order_id, payment_type)")
# order_reviews — filtered by review_score in analysis
spark.sql(f"OPTIMIZE {SILVER}.order_reviews ZORDER BY (order_id, review_score)")
# products — filtered by category
spark.sql(f"OPTIMIZE {SILVER}.products ZORDER BY (product_id, category_name_english)")
print("OPTIMIZE + Z-ORDER complete on Silver tables")

OPTIMIZE + Z-ORDER complete on Silver tables


In [0]:
# revenue_by_month — always filtered/ordered by year and month
spark.sql(f"OPTIMIZE {GOLD}.revenue_by_month ZORDER BY (year, month)")
# revenue_by_state — filtered by state
spark.sql(f"OPTIMIZE {GOLD}.revenue_by_state ZORDER BY (customer_state)")
# delivery_performance — filtered by year/month and late delivery
spark.sql(f"OPTIMIZE {GOLD}.delivery_performance ZORDER BY (year, month)")
# product_performance — filtered by category and sorted by revenue
spark.sql(f"OPTIMIZE {GOLD}.product_performance ZORDER BY (category_name_english, total_revenue)")
# customer_segments — filtered by customer_type and state
spark.sql(f"OPTIMIZE {GOLD}.customer_segments ZORDER BY (customer_type, customer_state)")
print("OPTIMIZE + Z-ORDER complete on Gold tables")

OPTIMIZE + Z-ORDER complete on Gold tables


In [0]:
tables_to_check = [
   f"{SILVER}.orders",
   f"{SILVER}.order_items",
   f"{GOLD}.revenue_by_month",
   f"{GOLD}.product_performance",
   f"{GOLD}.customer_segments"
]
for table in tables_to_check:
   history = spark.sql(f"DESCRIBE HISTORY {table} LIMIT 1")
   print(f"\n--- {table} ---")
   history.select("version", "timestamp", "operation", "operationMetrics").display(truncate=False)


--- olist_project.silver.orders ---


version,timestamp,operation,operationMetrics
0,2026-03-25T07:28:14.000Z,CREATE OR REPLACE TABLE AS SELECT,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 99441, numOutputBytes -> 6391063)"



--- olist_project.silver.order_items ---


version,timestamp,operation,operationMetrics
0,2026-03-25T07:29:27.000Z,CREATE OR REPLACE TABLE AS SELECT,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 112650, numOutputBytes -> 4101362)"



--- olist_project.gold.revenue_by_month ---


version,timestamp,operation,operationMetrics
1,2026-03-25T07:52:57.000Z,CREATE OR REPLACE TABLE AS SELECT,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 1664, numDeletionVectorsRemoved -> 0, numOutputRows -> 22, numOutputBytes -> 1664)"



--- olist_project.gold.product_performance ---


version,timestamp,operation,operationMetrics
0,2026-03-25T08:03:30.000Z,CREATE OR REPLACE TABLE AS SELECT,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 32216, numOutputBytes -> 855665)"



--- olist_project.gold.customer_segments ---


version,timestamp,operation,operationMetrics
0,2026-03-25T08:04:12.000Z,CREATE OR REPLACE TABLE AS SELECT,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 93395, numOutputBytes -> 3415542)"


In [0]:
silver_tables = ["orders", "customers", "order_items",
                "order_payments", "order_reviews", "products", "sellers"]
gold_tables   = ["revenue_by_month", "revenue_by_state", "delivery_performance",
                "product_performance", "customer_segments"]
for t in silver_tables:
   spark.sql(f"VACUUM {SILVER}.{t}")
   print(f"VACUUM complete — silver.{t}")
for t in gold_tables:
   spark.sql(f"VACUUM {GOLD}.{t}")
   print(f"VACUUM complete — gold.{t}")

VACUUM complete — silver.orders
VACUUM complete — silver.customers
VACUUM complete — silver.order_items
VACUUM complete — silver.order_payments
VACUUM complete — silver.order_reviews
VACUUM complete — silver.products
VACUUM complete — silver.sellers
VACUUM complete — gold.revenue_by_month
VACUUM complete — gold.revenue_by_state
VACUUM complete — gold.delivery_performance
VACUUM complete — gold.product_performance
VACUUM complete — gold.customer_segments


In [0]:
print("=== Delta Time Travel Demo ===\n")
# Show current version
current = spark.table(f"{SILVER}.orders")
print(f"Current version row count: {current.count()}")
# Read previous version (version 0 = raw Bronze ingestion)
previous = spark.read.format("delta") \
   .option("versionAsOf", 0) \
   .table(f"{SILVER}.orders")
print(f"Version 0 row count: {previous.count()}")
# Show full history
print("\n--- Full Delta History for silver.orders ---")
spark.sql(f"DESCRIBE HISTORY {SILVER}.orders") \
   .select("version", "timestamp", "operation", "operationParameters") \
   .display(truncate=False)

=== Delta Time Travel Demo ===

Current version row count: 99441
Version 0 row count: 99441

--- Full Delta History for silver.orders ---


version,timestamp,operation,operationParameters
2,2026-03-25T09:35:47.000Z,VACUUM END,Map(status -> COMPLETED)
1,2026-03-25T09:35:46.000Z,VACUUM START,"Map(retentionCheckEnabled -> true, defaultRetentionMillis -> 604800000)"
0,2026-03-25T07:28:14.000Z,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)"
